# Episode 22: A Plain HTML Frontend

AG-UI is powerful — but sometimes you just want a text box and a reply, and you own both ends. You don't need *any* framework for that.

**In this episode you'll build:** a chat backend in ~10 lines of Starlette, and a single-file HTML frontend.

## The design

The whole app is two pieces:

- **Backend** — one Starlette route. It receives a message, calls `agent.ask`, returns the reply as JSON.
- **Frontend** — one `index.html`. A text box and a `fetch()` call.

No protocol, no client library. Any language that can do an HTTP POST can talk to this agent.

## Setup

In [1]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from starlette.applications import Starlette
from starlette.routing import Route
from starlette.responses import JSONResponse

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

agent = Agent("assistant", prompt="You are a helpful assistant. Be concise.", config=config)


## The backend — and a real round-trip

The endpoint is one async function: read the JSON, `await agent.ask(...)`, return `reply.body`. We verify it immediately with Starlette's `TestClient`, which makes a real in-process HTTP request — no separate server needed to test it.

In [2]:
async def chat(request):
    data = await request.json()
    reply = await agent.ask(data["message"])
    return JSONResponse({"reply": reply.body})


app = Starlette(routes=[Route("/chat", chat, methods=["POST"])])

# Verify the endpoint with a real in-process HTTP request.
from starlette.testclient import TestClient

with TestClient(app) as client:
    response = client.post("/chat", json={"message": "Say hello in exactly three words."})
    print("status:", response.status_code)
    print("reply:", response.json()["reply"])


status: 200
reply: Hello there, friend!


## The frontend

A single HTML file — a text box, a button, and a `fetch` to `/chat`. Save it as `index.html`:

```html
<!doctype html>
<html>
<body>
  <h2>Chat with the agent</h2>
  <input id="msg" size="50" placeholder="Type a message...">
  <button onclick="send()">Send</button>
  <pre id="out"></pre>
  <script>
    async function send() {
      const message = document.getElementById('msg').value;
      const res = await fetch('/chat', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ message }),
      });
      const data = await res.json();
      document.getElementById('out').textContent = data.reply;
    }
  </script>
</body>
</html>
```

## Run it as a server

To serve the backend (and the HTML), save this as `plain_server.py` and run it from a terminal — `python plain_server.py`, then open `http://localhost:8000`.

```python
# plain_server.py  --  run from a terminal:  python plain_server.py
from dotenv import load_dotenv

load_dotenv()

import uvicorn
from starlette.applications import Starlette
from starlette.responses import FileResponse, JSONResponse
from starlette.routing import Route

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

agent = Agent("assistant", prompt="You are a helpful assistant. Be concise.",
              config=OpenAIConfig(model="gpt-4.1-mini"))


async def chat(request):
    data = await request.json()
    reply = await agent.ask(data["message"])
    return JSONResponse({"reply": reply.body})


async def index(request):
    return FileResponse("index.html")


app = Starlette(routes=[
    Route("/", index),
    Route("/chat", chat, methods=["POST"]),
])

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
```

## Additional content

**Streaming.** This endpoint returns the whole reply at once. To stream it token by token, subscribe a `MemoryStream` to `ModelMessageChunk` events (Episode 32) and return a `StreamingResponse` of server-sent events — the frontend reads them with an `EventSource`.

**Plain vs AG-UI.** A plain endpoint is the right call when you own both ends and want full control of the contract. Reach for AG-UI (Episode 21) when you want streaming, tool-call rendering, and HITL for free, and a standard client like CopilotKit.

**It's just HTTP.** Because the contract is plain JSON, a mobile app, a CLI, a cron job, or another service can all be clients — no AG2 on their side at all.

## What's next

Episode 23 closes Group 5 by opening the door to a whole ecosystem of external tools: the **Model Context Protocol**.